In [144]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [145]:
!pip install osmnx

In [146]:
import pandas as pd
import geopandas as gpd
import osmnx as ox
import networkx as nx
import numpy as np

from pyproj import Transformer
from sklearn.neighbors import BallTree
from pathlib import Path
import requests
import zipfile
from pathlib import Path

In [147]:
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/Movilidad_inteligente_madrid")

DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
GRAPH_DIR = DATA_DIR / "graphs"
VELOCIDADES_DIR = BASE_DIR / "mapa_velocidades"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
GRAPH_DIR.mkdir(parents=True, exist_ok=True)
VELOCIDADES_DIR.mkdir(parents=True, exist_ok=True)

ruta_velocidades = VELOCIDADES_DIR / "velocidades_madrid_final.geojson"

print("BASE_DIR:", BASE_DIR)
print("Existe GeoJSON velocidades:", ruta_velocidades.exists())
print("Ruta velocidades:", ruta_velocidades)

BASE_DIR: /content/drive/MyDrive/Movilidad_inteligente_madrid
Existe GeoJSON velocidades: True
Ruta velocidades: /content/drive/MyDrive/Movilidad_inteligente_madrid/mapa_velocidades/velocidades_madrid_final.geojson


In [148]:
from pathlib import Path
from google.colab import drive

# Montar Drive si todavía no está montado
if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

# Definir MYDRIVE
MYDRIVE = Path("/content/drive/MyDrive")

print("MYDRIVE:", MYDRIVE)
print("Existe:", MYDRIVE.exists())

MYDRIVE: /content/drive/MyDrive
Existe: True


In [149]:
resultados = list(MYDRIVE.rglob("velocidades_madrid_final.geojson"))

print("Archivos encontrados:", len(resultados))

for r in resultados:
    print(r)

Archivos encontrados: 1
/content/drive/MyDrive/Movilidad_inteligente_madrid/mapa_velocidades/velocidades_madrid_final.geojson


In [150]:
ruta_velocidades = resultados[0]

print("Ruta velocidades:", ruta_velocidades)
print("Existe:", ruta_velocidades.exists())

Ruta velocidades: /content/drive/MyDrive/Movilidad_inteligente_madrid/mapa_velocidades/velocidades_madrid_final.geojson
Existe: True


In [151]:
gdf_velocidades_full = gpd.read_file(ruta_velocidades)

print("Registros:", len(gdf_velocidades_full))
print("Columnas:")
print(gdf_velocidades_full.columns.tolist())

gdf_velocidades_full.head()

Registros: 271232
Columnas:
['u', 'v', 'key', 'highway', 'maxspeed', 'es_urbano', 'maxspeed_final', 'geometry']


,u,v,key,highway,maxspeed,es_urbano,maxspeed_final,geometry
0,21741584,21741587,0,[motorway],[100],False,[100],"LINESTRING (-3.69059 40.26517, -3.6905 40.2628..."
1,21741587,759985345,0,[motorway_link],"[40, 70]",False,"[40, 70]","LINESTRING (-3.69004 40.24808, -3.6901 40.2477..."
2,21741587,310025091,0,[motorway],[100],False,[100],"LINESTRING (-3.69004 40.24808, -3.68994 40.245..."
3,21741595,1317348460,0,"[motorway_link, tertiary]",[40],True,[40],"LINESTRING (-3.67595 40.1961, -3.6759 40.19589..."
4,21741595,310031145,0,[motorway],"[100, 120]",False,"[100, 120]","LINESTRING (-3.67595 40.1961, -3.67566 40.1956..."


In [152]:
from pathlib import Path
import zipfile

MYDRIVE = Path("/content/drive/MyDrive")

# zip descargado y se lee desde google drive
resultados_zip = list(MYDRIVE.rglob("202468-292-intensidad-trafico.zip"))

print("ZIP encontrados:", len(resultados_zip))

for r in resultados_zip:
    print(r)

if len(resultados_zip) == 0:
    raise FileNotFoundError("No se ha encontrado el ZIP en Google Drive.")

ruta_zip_medidores = resultados_zip[0]

#rutas
RAW_DIR = ruta_zip_medidores.parent
DATA_DIR = RAW_DIR.parent
BASE_DIR = DATA_DIR.parent
PROCESSED_DIR = DATA_DIR / "processed"
GRAPH_DIR = DATA_DIR / "graphs"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
GRAPH_DIR.mkdir(parents=True, exist_ok=True)

carpeta_medidores = RAW_DIR / "pmed_ubicacion"

# Crear carpeta de extracción
carpeta_medidores.mkdir(parents=True, exist_ok=True)

print("Ruta ZIP:", ruta_zip_medidores)
print("Carpeta extracción:", carpeta_medidores)

#contenido del zip
with zipfile.ZipFile(ruta_zip_medidores, "r") as zip_ref:
    print("Contenido del ZIP:")
    for nombre in zip_ref.namelist():
        print(nombre)

    # Extraer todo
    zip_ref.extractall(carpeta_medidores)

print("\nArchivos extraídos:")
for p in carpeta_medidores.iterdir():
    print(p.name)

# Buscar shapefile
shapefiles = list(carpeta_medidores.rglob("*.shp"))

print("\nShapefiles encontrados:", len(shapefiles))

for shp in shapefiles:
    print(shp)

if len(shapefiles) == 0:
    raise FileNotFoundError("No se ha encontrado ningún archivo .shp después de descomprimir.")

ruta_shp_medidores = shapefiles[0]

print("Usando shapefile:", ruta_shp_medidores)

ZIP encontrados: 1
/content/drive/MyDrive/Movilidad_inteligente_madrid/data/raw/202468-292-intensidad-trafico.zip
Ruta ZIP: /content/drive/MyDrive/Movilidad_inteligente_madrid/data/raw/202468-292-intensidad-trafico.zip
Carpeta extracción: /content/drive/MyDrive/Movilidad_inteligente_madrid/data/raw/pmed_ubicacion
Contenido del ZIP:
pmed_ubicacion_05-2026.cpg
pmed_ubicacion_05-2026.dbf
pmed_ubicacion_05-2026.prj
pmed_ubicacion_05-2026.shp
pmed_ubicacion_05-2026.shx

Archivos extraídos:
pmed_ubicacion_05-2026.cpg
pmed_ubicacion_05-2026.prj
pmed_ubicacion_05-2026.shp
pmed_ubicacion_05-2026.shx
pmed_ubicacion_05-2026.dbf

Shapefiles encontrados: 1
/content/drive/MyDrive/Movilidad_inteligente_madrid/data/raw/pmed_ubicacion/pmed_ubicacion_05-2026.shp
Usando shapefile: /content/drive/MyDrive/Movilidad_inteligente_madrid/data/raw/pmed_ubicacion/pmed_ubicacion_05-2026.shp


In [153]:
gdf_medidores = gpd.read_file(ruta_shp_medidores)

print("Filas:", len(gdf_medidores))
print("Columnas:", gdf_medidores.columns.tolist())

gdf_medidores.head()

Filas: 5072
Columnas: ['TIPO_ELEM', 'DISTRITO', 'ID', 'COD_CENT', 'NOMBRE', 'UTM_X', 'UTM_Y', 'LONGITUD', 'LATITUD', 'geometry']


,TIPO_ELEM,DISTRITO,ID,COD_CENT,NOMBRE,UTM_X,UTM_Y,LONGITUD,LATITUD,geometry
0,other,1.0,6835,18RA28PM01,18RA28PM01,438764.313318,4.474327e+06,-3.721794,40.417315,"POLYGON ((438768.544 4474328.066, 438766.024 4..."
1,other,9.0,1012,18RA66PM01,18RA66PM01,438740.943152,4.474610e+06,-3.722097,40.419861,"POLYGON ((438738.316 4474613.231, 438738.979 4..."
2,URB,10.0,5035,95013,FRUELA N-S,438004.401612,4.473859e+06,-3.730705,40.413040,"POLYGON ((438001.652 4473855.413, 438004.268 4..."
3,URB,5.0,5579,61068,Potosi E-O - Bolivia-Víctor Andrés Belaunde,442420.642251,4.478696e+06,-3.679095,40.456932,"POLYGON ((442417.854 4478699.064, 442418.653 4..."
4,URB,5.0,5580,61069,Víctor Andrés Belaunde N-S - Cochabamba-Potosi,442366.670466,4.478601e+06,-3.679723,40.456073,"POLYGON ((442370.039 4478603.48, 442367.151 44..."


In [154]:
gdf_medidores.columns = gdf_medidores.columns.str.lower()

medidores = gdf_medidores[
    ["id", "nombre", "utm_x", "utm_y", "longitud", "latitud"]
].copy()

medidores = medidores.dropna(subset=["longitud", "latitud"])
medidores = medidores.drop_duplicates(subset=["id"])

medidores["latitud"] = medidores["latitud"].astype(float)
medidores["longitud"] = medidores["longitud"].astype(float)

print("Medidores:", len(medidores))
medidores.head()

Medidores: 5072


,id,nombre,utm_x,utm_y,longitud,latitud
0,6835,18RA28PM01,438764.313318,4.474327e+06,-3.721794,40.417315
1,1012,18RA66PM01,438740.943152,4.474610e+06,-3.722097,40.419861
2,5035,FRUELA N-S,438004.401612,4.473859e+06,-3.730705,40.413040
3,5579,Potosi E-O - Bolivia-Víctor Andrés Belaunde,442420.642251,4.478696e+06,-3.679095,40.456932
4,5580,Víctor Andrés Belaunde N-S - Cochabamba-Potosi,442366.670466,4.478601e+06,-3.679723,40.456073


In [155]:
medidores.dtypes

,0
id,int64
nombre,object
utm_x,float64
utm_y,float64
longitud,float64
latitud,float64


In [156]:
#calidad de datos
print("IDs duplicados:", medidores["id"].duplicated().sum())
print("Latitud nula:", medidores["latitud"].isna().sum())
print("Longitud nula:", medidores["longitud"].isna().sum())

fuera_madrid = medidores[
    ~(
        (medidores["latitud"].between(40.30, 40.55)) &
        (medidores["longitud"].between(-3.90, -3.50))
    )
]

print("Medidores fuera de rango Madrid:", len(fuera_madrid))
fuera_madrid.head()

IDs duplicados: 0
Latitud nula: 0
Longitud nula: 0
Medidores fuera de rango Madrid: 0


,id,nombre,utm_x,utm_y,longitud,latitud


In [157]:
medidores[["latitud", "longitud"]].describe()

,latitud,longitud
count,5072.000000,5072.000000
mean,40.430447,-3.684001
std,0.039163,0.042728
min,40.332454,-3.836886
25%,40.398978,-3.712553
50%,40.431302,-3.686923
75%,40.460080,-3.656194
max,40.515611,-3.551623


## Descarga de la red viaria de Madrid con OSMnx

In [158]:
ruta_grafo_osm_base = GRAPH_DIR / "red_osm_madrid_base.graphml"

if ruta_grafo_osm_base.exists():
    print("Cargando red OSM desde Drive")
    G_osm = ox.load_graphml(ruta_grafo_osm_base)
else:
    print("No existe red OSM. Descargando desde OSMnx...")
    G_osm = ox.graph_from_place(
        "Madrid, Spain",
        network_type="drive",
        simplify=True
    )
    ox.save_graphml(G_osm, filepath=ruta_grafo_osm_base)

print("Nodos OSM:", G_osm.number_of_nodes())
print("Aristas OSM:", G_osm.number_of_edges())
print("Ruta GraphML:", ruta_grafo_osm_base)

Cargando red OSM desde Drive
Nodos OSM: 31443
Aristas OSM: 61843
Ruta GraphML: /content/drive/MyDrive/Movilidad_inteligente_madrid/data/graphs/red_osm_madrid_base.graphml


## Snap de medidores a la red OSM

Cada punto medidor se asocia al nodo OSM más cercano.  
 `snap to graph + camino más corto sobre OSMnx`.

In [159]:
osm_nodes, distancias = ox.distance.nearest_nodes(
    G_osm,
    X=medidores["longitud"].values,
    Y=medidores["latitud"].values,
    return_dist=True
)

medidores["osm_node"] = osm_nodes
medidores["distancia_osm_node_m"] = distancias

medidores.head()

,id,nombre,utm_x,utm_y,longitud,latitud,osm_node,distancia_osm_node_m
0,6835,18RA28PM01,438764.313318,4.474327e+06,-3.721794,40.417315,32636471,79.621034
1,1012,18RA66PM01,438740.943152,4.474610e+06,-3.722097,40.419861,315259372,54.829432
2,5035,FRUELA N-S,438004.401612,4.473859e+06,-3.730705,40.413040,305399713,15.640852
3,5579,Potosi E-O - Bolivia-Víctor Andrés Belaunde,442420.642251,4.478696e+06,-3.679095,40.456932,1672792326,40.116809
4,5580,Víctor Andrés Belaunde N-S - Cochabamba-Potosi,442366.670466,4.478601e+06,-3.679723,40.456073,119794656,14.979371


In [160]:
medidores["distancia_osm_node_m"].describe()

,distancia_osm_node_m
count,5072.000000
mean,36.615560
std,28.020252
min,0.101869
25%,16.220982
50%,28.055324
75%,50.402014
max,345.242185


In [161]:
medidores.sort_values("distancia_osm_node_m", ascending=False).head(20)

,id,nombre,utm_x,utm_y,longitud,latitud,osm_node,distancia_osm_node_m
1938,5268,(TACTICO)SALIDA POLIGONO N-S,434514.277528,4.468632e+06,-3.771300,40.365688,306400716,345.242185
107,4928,(TACTICO) AV. POBLADOS O-E (GIRO A ERICA),435668.604184,4.470451e+06,-3.757889,40.382164,282940163,228.695575
1394,4960,(TACTICO) ERICA N-S (CENTRO C.I.E.),435691.441140,4.470458e+06,-3.757621,40.382233,282940163,219.565346
1760,11199,Fuerzas Armadas - Ciudad Deportiva O-E - Fuerz...,448191.892131,4.481464e+06,-3.611259,40.482247,1012899036,178.720804
2325,11200,Fuerzas Armadas - Ciudad Deportiva O-E (Vía Se...,448192.910496,4.481435e+06,-3.611244,40.481993,1012899118,178.333987
4888,6876,12XC06PM01,441861.911399,4.471142e+06,-3.684994,40.388851,317771984,169.480943
1745,11191,"Av Fuerzas Armadas, 322 O-E - Av Fuerzas Armad...",447371.205461,4.481464e+06,-3.620941,40.482198,969169634,168.795375
1759,11192,"Av Fuerzas Armadas, 322 O-E (Via Servicio) - A...",447372.223825,4.481436e+06,-3.620927,40.481944,969169634,168.326941
446,9916,SINESIO DELGADO O-E (HOSPITAL CARLOS III-ENTRA...,440954.801598,4.480675e+06,-3.696566,40.474658,26205041,163.715082
445,9915,SINESIO DELGADO E-O (SALIDA TUNEL-HOSPITAL CAR...,440949.748176,4.480680e+06,-3.696627,40.474708,26205041,156.229910


## Análisis de distancias entre medidores y generación de pares candidatos

In [162]:
# Coordenadas en radianes para distancia haversine
coords = np.radians(medidores[["latitud", "longitud"]].values)

tree = BallTree(coords, metric="haversine")

# Calculamos hasta los 20 vecinos más cercanos para estudiar la distribución
K_ANALISIS = 20
distancias, indices = tree.query(coords, k=K_ANALISIS + 1)

R = 6371000  # radio tierra metros
distancias_m = distancias * R

In [163]:
resumen_vecinos = pd.DataFrame({
    "vecino_1_m": distancias_m[:, 1],
    "vecino_3_m": distancias_m[:, 3],
    "vecino_5_m": distancias_m[:, 5],
    "vecino_10_m": distancias_m[:, 10],
    "vecino_20_m": distancias_m[:, 20],
})

resumen_vecinos.describe(percentiles=[0.25, 0.5, 0.75, 0.90, 0.95])

,vecino_1_m,vecino_3_m,vecino_5_m,vecino_10_m,vecino_20_m
count,5072.000000,5072.000000,5072.000000,5072.000000,5072.000000
mean,60.299512,134.802462,192.218004,303.847693,479.846961
std,50.277204,83.244571,106.243956,150.109013,302.088483
min,0.000000,10.896282,15.656927,80.867585,195.699433
25%,16.317860,89.649261,130.094414,216.112762,344.834643
50%,51.730719,125.062705,172.480334,274.661836,421.074841
75%,90.709754,164.091985,225.365385,348.903363,527.182705
90%,123.780820,211.121938,299.572190,449.004668,660.410756
95%,148.631821,252.537244,367.262305,544.203109,817.526027
max,559.979456,1626.513574,1656.200417,1741.671511,4377.116647


In [164]:
radio_candidatos_m = np.percentile(distancias_m[:, 5], 90)

print("Radio de candidatos calculado:", radio_candidatos_m, "metros")

Radio de candidatos calculado: 299.57218999318764 metros


In [165]:
radio_candidatos_rad = radio_candidatos_m / R

indices_radio, distancias_radio = tree.query_radius(
    coords,
    r=radio_candidatos_rad,
    return_distance=True,
    sort_results=True
)

pares_candidatos = []

for i in range(len(medidores)):
    medidor_origen = medidores.iloc[i]

    for j, distancia_rad in zip(indices_radio[i], distancias_radio[i]):
        if i == j:
            continue

        medidor_destino = medidores.iloc[j]

        pares_candidatos.append({
            "id_origen": medidor_origen["id"],
            "id_destino": medidor_destino["id"],
            "distancia_directa_m": distancia_rad * R,
            "osm_node_origen": medidor_origen["osm_node"],
            "osm_node_destino": medidor_destino["osm_node"]
        })

df_pares = pd.DataFrame(pares_candidatos)

print("Número de pares candidatos:", len(df_pares))
df_pares.head()

Número de pares candidatos: 60494


,id_origen,id_destino,distancia_directa_m,osm_node_origen,osm_node_destino
0,6835,6833,20.104431,32636471,32636471
1,6835,6836,91.872471,32636471,315261895
2,6835,6837,93.934596,32636471,315264896
3,6835,6827,110.969150,32636471,315261895
4,6835,1042,116.845927,32636471,315265031


In [166]:
candidatos_por_medidor = (
    df_pares
    .groupby("id_origen")
    .size()
    .reset_index(name="num_candidatos")
)

candidatos_por_medidor["num_candidatos"].describe()


,num_candidatos
count,5060.000000
mean,11.955336
std,6.228532
min,1.000000
25%,7.000000
50%,11.000000
75%,16.000000
max,38.000000


## Cálculo de caminos reales sobre la red OSM

In [167]:
from tqdm import tqdm
aristas_reales = []

for _, row in tqdm(df_pares.iterrows(), total=len(df_pares)):
    id_origen = int(row["id_origen"])
    id_destino = int(row["id_destino"])

    osm_origen = int(row["osm_node_origen"])
    osm_destino = int(row["osm_node_destino"])

    distancia_directa_m = float(row["distancia_directa_m"])

    # Caso especial: dos medidores asociados al mismo nodo OSM
    if osm_origen == osm_destino:
        aristas_reales.append({
            "id_origen": id_origen,
            "id_destino": id_destino,
            "osm_node_origen": osm_origen,
            "osm_node_destino": osm_destino,
            "distancia_directa_m": distancia_directa_m,
            "distancia_red_m": 0.0,
            "tipo_conexion": "mismo_nodo_osm"
        })
        continue

    try:
        distancia_red_m = nx.shortest_path_length(
            G_osm,
            source=osm_origen,
            target=osm_destino,
            weight="length"
        )

        aristas_reales.append({
            "id_origen": id_origen,
            "id_destino": id_destino,
            "osm_node_origen": osm_origen,
            "osm_node_destino": osm_destino,
            "distancia_directa_m": distancia_directa_m,
            "distancia_red_m": float(distancia_red_m),
            "tipo_conexion": "camino_osm"
        })

    except (nx.NetworkXNoPath, nx.NodeNotFound):
        # Si no hay camino real en OSM, no se crea arista
        continue

df_aristas_reales = pd.DataFrame(aristas_reales).drop_duplicates()

print("Pares candidatos:", len(df_pares))
print("Aristas reales encontradas:", len(df_aristas_reales))

df_aristas_reales.head()

100%|██████████| 60494/60494 [01:42<00:00, 589.03it/s]


Pares candidatos: 60494
Aristas reales encontradas: 60251


,id_origen,id_destino,osm_node_origen,osm_node_destino,distancia_directa_m,distancia_red_m,tipo_conexion
0,6835,6833,32636471,32636471,20.104431,0.000000,mismo_nodo_osm
1,6835,6836,32636471,315261895,91.872471,6655.017406,camino_osm
2,6835,6837,32636471,315264896,93.934596,2248.920678,camino_osm
3,6835,6827,32636471,315261895,110.969150,6655.017406,camino_osm
4,6835,1042,32636471,315265031,116.845927,2583.076553,camino_osm


## Validación de aristas
Se calcula el factor de rodeo:

\[
factor\_rodeo = \frac{distancia\_red\_m}{distancia\_directa\_m}
\]

Este factor permite detectar conexiones donde dos medidores están cerca en línea recta, pero el camino real por carretera es mucho más largo.  
Estas aristas no se eliminan automáticamente: se marcan para revisión.

In [168]:
df_aristas_reales["factor_rodeo"] = np.where(
    df_aristas_reales["distancia_directa_m"] > 0,
    df_aristas_reales["distancia_red_m"] / df_aristas_reales["distancia_directa_m"],
    np.nan
)

# Si están en el mismo nodo OSM, consideramos factor de rodeo 0
df_aristas_reales.loc[
    df_aristas_reales["tipo_conexion"] == "mismo_nodo_osm",
    "factor_rodeo"
] = 0

df_aristas_reales[[
    "distancia_directa_m",
    "distancia_red_m",
    "factor_rodeo"
]].describe(percentiles=[0.5, 0.75, 0.90, 0.95, 0.99])

,distancia_directa_m,distancia_red_m,factor_rodeo
count,60251.000000,60251.000000,60251.000000
mean,182.030328,633.999288,4.182369
std,77.679277,1100.877076,12.497009
min,0.000000,0.000000,0.000000
50%,190.990925,332.662946,1.647003
75%,248.275283,588.922277,3.168965
90%,279.856331,1309.787701,7.471431
95%,289.919282,2561.422004,14.427170
99%,297.865966,5251.550964,47.340847
max,299.571214,13963.539403,734.950413


In [169]:
# Umbrales
umbral_distancia_red = df_aristas_reales["distancia_red_m"].quantile(0.95)
umbral_rodeo = df_aristas_reales["factor_rodeo"].quantile(0.95)

print("Umbral distancia red P95:", umbral_distancia_red)
print("Umbral rodeo P95:", umbral_rodeo)

Umbral distancia red P95: 2561.4220035395583
Umbral rodeo P95: 14.427170031331508


In [170]:
df_aristas_reales["revisar_distancia_red_alta"] = (
    df_aristas_reales["distancia_red_m"] > umbral_distancia_red
)

df_aristas_reales["revisar_rodeo_alto"] = (
    df_aristas_reales["factor_rodeo"] > umbral_rodeo
)

def clasificar_arista(row):
    if row["revisar_distancia_red_alta"] and row["revisar_rodeo_alto"]:
        return "revisar_distancia_y_rodeo"
    elif row["revisar_rodeo_alto"]:
        return "revisar_rodeo_alto"
    elif row["revisar_distancia_red_alta"]:
        return "revisar_distancia_red_alta"
    else:
        return "ok"

df_aristas_reales["calidad_arista"] = df_aristas_reales.apply(clasificar_arista, axis=1)

df_aristas_reales["calidad_arista"].value_counts()

,count
calidad_arista,
ok,56488
revisar_distancia_y_rodeo,2263
revisar_rodeo_alto,750
revisar_distancia_red_alta,750


In [171]:
# Aristas con mayor factor de rodeo
df_aristas_reales.sort_values("factor_rodeo", ascending=False).head(20)

,id_origen,id_destino,osm_node_origen,osm_node_destino,distancia_directa_m,distancia_red_m,tipo_conexion,factor_rodeo,revisar_distancia_red_alta,revisar_rodeo_alto,calidad_arista
57566,6952,6949,5360984763,307997125,12.250197,9003.287558,camino_osm,734.950413,True,True,revisar_distancia_y_rodeo
46633,1015,1016,338920029,21723233,8.791932,6123.576228,camino_osm,696.499499,True,True,revisar_distancia_y_rodeo
9513,11370,6786,25549913,297767512,13.582513,6413.325810,camino_osm,472.175201,True,True,revisar_distancia_y_rodeo
57567,6952,6950,5360984763,307997125,19.560019,9003.287558,camino_osm,460.290317,True,True,revisar_distancia_y_rodeo
9540,1052,1049,388087148,2493682299,25.653528,11766.287184,camino_osm,458.661561,True,True,revisar_distancia_y_rodeo
57569,6951,6949,5360984763,307997125,19.911916,9003.287558,camino_osm,452.155757,True,True,revisar_distancia_y_rodeo
1162,11373,11374,9823064711,429461526,18.481650,6765.444483,camino_osm,366.062801,True,True,revisar_distancia_y_rodeo
56893,3826,6790,21525883,20953256,6.961135,2341.325133,camino_osm,336.342442,False,True,revisar_rodeo_alto
1163,11373,11375,9823064711,429461526,21.174514,6765.444483,camino_osm,319.508849,True,True,revisar_distancia_y_rodeo
26829,6738,6737,25938853,2537144683,18.511306,5833.573622,camino_osm,315.135710,True,True,revisar_distancia_y_rodeo


In [172]:
# Aristas con mayor distancia real sobre red
df_aristas_reales.sort_values("distancia_red_m", ascending=False).head(20)

,id_origen,id_destino,osm_node_origen,osm_node_destino,distancia_directa_m,distancia_red_m,tipo_conexion,factor_rodeo,revisar_distancia_red_alta,revisar_rodeo_alto,calidad_arista
10262,11496,6858,315244935,2136384930,209.732125,13963.539403,camino_osm,66.577971,True,True,revisar_distancia_y_rodeo
10259,11496,6857,315244935,2136384930,208.098671,13963.539403,camino_osm,67.100570,True,True,revisar_distancia_y_rodeo
10270,11496,1032,315244935,2136384930,231.140447,13963.539403,camino_osm,60.411493,True,True,revisar_distancia_y_rodeo
10272,11496,7118,315244935,2136384930,239.828567,13963.539403,camino_osm,58.223003,True,True,revisar_distancia_y_rodeo
10269,11496,7145,315244935,2136384930,219.215723,13963.539403,camino_osm,63.697709,True,True,revisar_distancia_y_rodeo
10268,11496,6856,315244935,2136384930,218.721078,13963.539403,camino_osm,63.841764,True,True,revisar_distancia_y_rodeo
49886,1035,1032,315244935,2136384930,227.371704,13963.539403,camino_osm,61.412828,True,True,revisar_distancia_y_rodeo
49884,1035,7145,315244935,2136384930,216.587352,13963.539403,camino_osm,64.470706,True,True,revisar_distancia_y_rodeo
49876,1035,6857,315244935,2136384930,204.252481,13963.539403,camino_osm,68.364112,True,True,revisar_distancia_y_rodeo
49877,1035,6858,315244935,2136384930,206.291569,13963.539403,camino_osm,67.688367,True,True,revisar_distancia_y_rodeo


## Construcción grafo

In [173]:
G_medidores = nx.DiGraph()

# Añadir todos los medidores reales como nodos
for _, row in medidores.iterrows():
    G_medidores.add_node(
        int(row["id"]),
        nombre=row["nombre"],
        latitud=float(row["latitud"]),
        longitud=float(row["longitud"]),
        osm_node=int(row["osm_node"]),
        distancia_osm_node_m=float(row["distancia_osm_node_m"])
    )

# Añadir aristas reales calculadas sobre OSM
for _, row in df_aristas_reales.iterrows():
    G_medidores.add_edge(
        int(row["id_origen"]),
        int(row["id_destino"]),
        distancia_directa_m=float(row["distancia_directa_m"]),
        distancia_red_m=float(row["distancia_red_m"]),
        factor_rodeo=float(row["factor_rodeo"]),
        weight=float(row["distancia_red_m"]),
        tipo_conexion=row["tipo_conexion"],
        calidad_arista=row["calidad_arista"]
    )

print("Nodos:", G_medidores.number_of_nodes())
print("Aristas:", G_medidores.number_of_edges())

Nodos: 5072
Aristas: 60251


In [174]:
nodos_con_aristas = set(df_aristas_reales["id_origen"]).union(
    set(df_aristas_reales["id_destino"])
)

nodos_aislados = set(medidores["id"]) - nodos_con_aristas

print("Medidores totales:", medidores["id"].nunique())
print("Medidores con alguna arista:", len(nodos_con_aristas))
print("Medidores aislados:", len(nodos_aislados))

print("Componentes débiles:", nx.number_weakly_connected_components(G_medidores))
print("Componentes fuertes:", nx.number_strongly_connected_components(G_medidores))

Medidores totales: 5072
Medidores con alguna arista: 5060
Medidores aislados: 12
Componentes débiles: 79
Componentes fuertes: 98


In [175]:
medidores_aislados = medidores[
    medidores["id"].isin(nodos_aislados)
].copy()

medidores_aislados[
    ["id", "nombre", "latitud", "longitud", "osm_node", "distancia_osm_node_m"]
].sort_values("distancia_osm_node_m", ascending=False)

,id,nombre,latitud,longitud,osm_node,distancia_osm_node_m
1938,5268,(TACTICO)SALIDA POLIGONO N-S,40.365688,-3.771300,306400716,345.242185
345,10210,PM43041,40.515611,-3.685133,2590772533,101.383315
4388,6584,(TACTICO) SALIDA CUARTEL ARTILLERIA,40.513409,-3.679176,255961297,90.867682
2302,6489,Embajadores - Santa Catalina-Carretera Villave...,40.369021,-3.676004,306101165,90.843345
2761,5160,(TACTICO)JOSE CADALSO S-N(VALLE INCLAN-AV. LAS...,40.382518,-3.771977,26085559,35.193999
872,6923,San Cipriano - Efigencia-Caños San Pedro,40.403851,-3.601293,307534056,20.827393
4743,3528,Tumaco - Tumaco-Tampico,40.445069,-3.635408,114123521,20.281911
675,5298,(TACTICO)ALLARIZ O-E(PROGRESO-AV. CARABANCHEL ...,40.367476,-3.753622,306163498,16.163931
3214,4868,(TACTICO) BATALLA GARELLANO Nº 27 S-N (SIRRACH...,40.455164,-3.793131,292702537,15.418926
3540,10012,(TACTICO) Salida Clinica Lopez Ibor,40.467961,-3.724342,4777894121,13.405010


## VISUALIZACIÓN DE NODOS EN EL MAPA

In [176]:
import folium
from folium.plugins import MarkerCluster

# Centro del mapa en la media de coordenadas de los medidores
lat_centro = medidores["latitud"].mean()
lon_centro = medidores["longitud"].mean()

mapa = folium.Map(
    location=[lat_centro, lon_centro],
    zoom_start=13,
    tiles="CartoDB positron"   # fondo claro, las aristas se ven mejor
)

# Verde  → medidor con aristas OK
# Rojo   → medidor aislado (sin aristas)

nodos_con_aristas_viz = set(df_aristas_reales["id_origen"]).union(
    set(df_aristas_reales["id_destino"])
)

cluster = MarkerCluster(name="Puntos medidores", show=True)

for _, row in medidores.iterrows():
    es_aislado = int(row["id"]) not in nodos_con_aristas_viz
    color = "red" if es_aislado else "green"
    icono  = "times" if es_aislado else "circle"

    folium.CircleMarker(
        location=[row["latitud"], row["longitud"]],
        radius=5,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.8,
        popup=folium.Popup(
            f"<b>ID:</b> {int(row['id'])}<br>"
            f"<b>Nombre:</b> {row['nombre']}<br>"
            f"<b>OSM node:</b> {int(row['osm_node'])}<br>"
            f"<b>Dist. a OSM:</b> {row['distancia_osm_node_m']:.1f} m<br>"
            f"<b>Estado:</b> {'🔴 Aislado' if es_aislado else '🟢 Conectado'}",
            max_width=250
        ),
        tooltip=f"ID {int(row['id'])} – {row['nombre']}"
    ).add_to(cluster)

cluster.add_to(mapa)

In [177]:
coord_por_id = medidores.set_index("id")[["latitud", "longitud"]].to_dict("index")

# Paleta de colores por calidad de arista
COLOR_ARISTA = {
    "ok":                         "#2ecc71",   # verde
    "revisar_rodeo_alto":         "#f39c12",   # naranja
    "revisar_distancia_red_alta": "#3498db",   # azul
    "revisar_distancia_y_rodeo":  "#e74c3c",   # rojo
    "mismo_nodo_osm":             "#9b59b6",   # morado
}

capa_ok      = folium.FeatureGroup(name="Aristas OK",            show=True)
capa_rodeo   = folium.FeatureGroup(name="Revisar rodeo",         show=True)
capa_dist    = folium.FeatureGroup(name="Revisar distancia",     show=True)
capa_ambos   = folium.FeatureGroup(name="Revisar dist+rodeo",   show=False)
capa_mismo   = folium.FeatureGroup(name="Mismo nodo OSM",        show=False)

CAPA_MAP = {
    "ok":                         capa_ok,
    "revisar_rodeo_alto":         capa_rodeo,
    "revisar_distancia_red_alta": capa_dist,
    "revisar_distancia_y_rodeo":  capa_ambos,
    "mismo_nodo_osm":             capa_mismo,
}

for _, row in df_aristas_reales.iterrows():
    id_o = int(row["id_origen"])
    id_d = int(row["id_destino"])

    if id_o not in coord_por_id or id_d not in coord_por_id:
        continue

    p_origen  = (coord_por_id[id_o]["latitud"], coord_por_id[id_o]["longitud"])
    p_destino = (coord_por_id[id_d]["latitud"], coord_por_id[id_d]["longitud"])

    calidad = row["calidad_arista"]
    color   = COLOR_ARISTA.get(calidad, "#95a5a6")
    capa    = CAPA_MAP.get(calidad, capa_ok)

    folium.PolyLine(
        locations=[p_origen, p_destino],
        color=color,
        weight=1.5,
        opacity=0.6,
        tooltip=(
            f"ID {id_o} → {id_d} | "
            f"Red: {row['distancia_red_m']:.0f} m | "
            f"Rodeo: {row['factor_rodeo']:.2f}x | "
            f"{calidad}"
        )
    ).add_to(capa)

for capa in [capa_ok, capa_rodeo, capa_dist, capa_ambos, capa_mismo]:
    capa.add_to(mapa)

In [178]:
# Análisis de componentes débiles del grafo de medidores

componentes = list(nx.weakly_connected_components(G_medidores))
componentes_ordenados = sorted(componentes, key=len, reverse=True)

print("Total de componentes débiles:", len(componentes_ordenados))
print()
print("Tamaño de cada componente:")

for i, comp in enumerate(componentes_ordenados):
    print(f"Componente {i + 1}: {len(comp)} nodos")

Total de componentes débiles: 79

Tamaño de cada componente:
Componente 1: 4223 nodos
Componente 2: 146 nodos
Componente 3: 109 nodos
Componente 4: 64 nodos
Componente 5: 60 nodos
Componente 6: 52 nodos
Componente 7: 43 nodos
Componente 8: 29 nodos
Componente 9: 25 nodos
Componente 10: 19 nodos
Componente 11: 15 nodos
Componente 12: 15 nodos
Componente 13: 13 nodos
Componente 14: 12 nodos
Componente 15: 12 nodos
Componente 16: 11 nodos
Componente 17: 11 nodos
Componente 18: 10 nodos
Componente 19: 8 nodos
Componente 20: 8 nodos
Componente 21: 8 nodos
Componente 22: 7 nodos
Componente 23: 7 nodos
Componente 24: 7 nodos
Componente 25: 6 nodos
Componente 26: 5 nodos
Componente 27: 5 nodos
Componente 28: 5 nodos
Componente 29: 5 nodos
Componente 30: 5 nodos
Componente 31: 5 nodos
Componente 32: 4 nodos
Componente 33: 4 nodos
Componente 34: 4 nodos
Componente 35: 4 nodos
Componente 36: 4 nodos
Componente 37: 4 nodos
Componente 38: 4 nodos
Componente 39: 4 nodos
Componente 40: 4 nodos
Compon

In [179]:
# Ver qué zonas son los componentes más grandes fuera del componente principal

comp_principal = componentes_ordenados[0]

print("Componentes desconectados más grandes fuera del principal:")
print()

for i, comp in enumerate(componentes_ordenados[1:11], start=2):
    nodos_comp = list(comp)

    lats = [G_medidores.nodes[n]["latitud"] for n in nodos_comp]
    lons = [G_medidores.nodes[n]["longitud"] for n in nodos_comp]
    nombres = [G_medidores.nodes[n]["nombre"] for n in nodos_comp[:3]]

    print(f"Componente {i} ({len(comp)} nodos)")
    print(f"Centro aproximado: lat {np.mean(lats):.4f}, lon {np.mean(lons):.4f}")
    print(f"Ejemplos: {nombres}")
    print()

Componentes desconectados más grandes fuera del principal:

Componente 2 (146 nodos)
Centro aproximado: lat 40.3880, lon -3.7661
Ejemplos: ['ILLESCAS ENTRE Pº EXTREMADURA Y SESEÑA SENTIDO O-E', '(TACTICO)ESCALONA O-E(MAQUEDA-ILLESCAS)', 'AV.POBLADOS N-S(SEBASTIAN ALVAROMELILOTO)']

Componente 3 (109 nodos)
Centro aproximado: lat 40.5043, lon -3.6983
Ejemplos: ['MONASTERIO SILOS O-E (MONASTERIO DE CAAVEIRO - MONASTERIO SAMOS)', 'MONASTERIO SAMOS N-S (MONASTERIO ESCORIAL - MONASTERIO SILOS)', 'MONASTERIO SAMOS S-N (MONASTERIO SUSO Y YUSO - MONASTERIO SILOS)']

Componente 4 (64 nodos)
Centro aproximado: lat 40.4571, lon -3.7827
Ejemplos: ['(MICRO) ANA TERESA, 29A O-E (A. VINDEL - PLEYADES)', 'AV. OSA MAYOR Ø73 O-E (CAROLI-PEREZ DE VICTORIA)', '(TACTICO) ARANDIGA Nº 8 N-S (ARASCUES - AV. OSA MAYOR)']

Componente 5 (60 nodos)
Centro aproximado: lat 40.4910, lon -3.6123
Ejemplos: ['Av. Fuerzas Armadas (CARRIL BUS) - Ciudad Deportiva - Gta. Antoñete', 'Acceso Gta. Antoñete (Carril BUS) - Jose

In [180]:
# Distancia mínima entre los componentes desconectados y el componente principal

print("Distancia mínima entre componentes desconectados y el componente principal:")
print()

nodos_principal = list(comp_principal)

coords_principal = np.radians([
    [
        G_medidores.nodes[n]["latitud"],
        G_medidores.nodes[n]["longitud"]
    ]
    for n in nodos_principal
])

tree_principal = BallTree(coords_principal, metric="haversine")

for i, comp in enumerate(componentes_ordenados[1:11], start=2):
    nodos_comp = list(comp)

    coords_comp = np.radians([
        [
            G_medidores.nodes[n]["latitud"],
            G_medidores.nodes[n]["longitud"]
        ]
        for n in nodos_comp
    ])

    dists, _ = tree_principal.query(coords_comp, k=1)
    dist_min = dists.min() * 6371000

    print(
        f"Componente {i} ({len(comp)} nodos): "
        f"distancia mínima al principal = {dist_min:.0f} m"
    )

Distancia mínima entre componentes desconectados y el componente principal:

Componente 2 (146 nodos): distancia mínima al principal = 337 m
Componente 3 (109 nodos): distancia mínima al principal = 413 m
Componente 4 (64 nodos): distancia mínima al principal = 3093 m
Componente 5 (60 nodos): distancia mínima al principal = 1547 m
Componente 6 (52 nodos): distancia mínima al principal = 326 m
Componente 7 (43 nodos): distancia mínima al principal = 1174 m
Componente 8 (29 nodos): distancia mínima al principal = 828 m
Componente 9 (25 nodos): distancia mínima al principal = 848 m
Componente 10 (19 nodos): distancia mínima al principal = 623 m
Componente 11 (15 nodos): distancia mínima al principal = 311 m


In [181]:
# Guardar resumen de componentes

resumen_componentes = []

for i, comp in enumerate(componentes_ordenados):
    nodos_comp = list(comp)

    lats = [G_medidores.nodes[n]["latitud"] for n in nodos_comp]
    lons = [G_medidores.nodes[n]["longitud"] for n in nodos_comp]

    if i == 0:
        dist_min_principal = 0
    else:
        coords_comp = np.radians([
            [
                G_medidores.nodes[n]["latitud"],
                G_medidores.nodes[n]["longitud"]
            ]
            for n in nodos_comp
        ])

        dists, _ = tree_principal.query(coords_comp, k=1)
        dist_min_principal = round(dists.min() * 6371000, 1)

    resumen_componentes.append({
        "componente": i + 1,
        "num_nodos": len(comp),
        "lat_centro": round(np.mean(lats), 4),
        "lon_centro": round(np.mean(lons), 4),
        "dist_min_al_principal_m": dist_min_principal,
        "es_principal": i == 0
    })

df_componentes = pd.DataFrame(resumen_componentes)

df_componentes.to_csv(
    PROCESSED_DIR / "analisis_componentes.csv",
    index=False,
    encoding="utf-8"
)

print("Guardado en:", PROCESSED_DIR / "analisis_componentes.csv")
df_componentes.head(15)

Guardado en: /content/drive/MyDrive/Movilidad_inteligente_madrid/data/processed/analisis_componentes.csv


,componente,num_nodos,lat_centro,lon_centro,dist_min_al_principal_m,es_principal
0,1,4223,40.4284,-3.6820,0.0,True
1,2,146,40.3880,-3.7661,337.5,False
2,3,109,40.5043,-3.6983,413.3,False
3,4,64,40.4571,-3.7827,3093.1,False
4,5,60,40.4910,-3.6123,1547.0,False
5,6,52,40.4927,-3.7249,325.9,False
6,7,43,40.3672,-3.6045,1173.7,False
7,8,29,40.4070,-3.6117,828.1,False
8,9,25,40.4827,-3.6204,847.7,False
9,10,19,40.4762,-3.7409,623.1,False


In [182]:
# Mapa de componentes

mapa_componentes = folium.Map(
    location=[
        medidores["latitud"].mean(),
        medidores["longitud"].mean()
    ],
    zoom_start=11
)

# Colores para los primeros componentes
colores = [
    "blue", "red", "green", "purple", "orange",
    "darkred", "lightred", "beige", "darkblue", "darkgreen",
    "cadetblue", "pink", "lightblue", "lightgreen", "gray"
]

# Pintamos solo los primeros 15 componentes para que el mapa sea legible
for i, comp in enumerate(componentes_ordenados[:15]):
    color = colores[i % len(colores)]
    etiqueta = "Principal" if i == 0 else f"Comp {i + 1} ({len(comp)} nodos)"

    for nodo in comp:
        lat = G_medidores.nodes[nodo]["latitud"]
        lon = G_medidores.nodes[nodo]["longitud"]
        nombre = G_medidores.nodes[nodo]["nombre"]

        folium.CircleMarker(
            location=[lat, lon],
            radius=3 if i == 0 else 5,
            color=color,
            fill=True,
            fill_opacity=0.6 if i == 0 else 0.9,
            popup=f"[{etiqueta}]<br>ID: {nodo}<br>{nombre}"
        ).add_to(mapa_componentes)

ruta_mapa_componentes = PROCESSED_DIR / "mapa_componentes.html"
mapa_componentes.save(ruta_mapa_componentes)

print("Mapa guardado en:", ruta_mapa_componentes.resolve())

Mapa guardado en: /content/drive/MyDrive/Movilidad_inteligente_madrid/data/processed/mapa_componentes.html


In [183]:
from IPython.display import display

display(mapa_componentes)

## Integración de velocidades en el grafo



In [184]:
from pathlib import Path
import geopandas as gpd

BASE_DIR = Path("/content/drive/MyDrive/Movilidad_inteligente_madrid")
VELOCIDADES_DIR = BASE_DIR / "mapa_velocidades"

ruta_velocidades = VELOCIDADES_DIR / "velocidades_madrid_final.geojson"

print("Ruta velocidades:", ruta_velocidades)
print("Existe:", ruta_velocidades.exists())

gdf_velocidades_full = gpd.read_file(ruta_velocidades)

print("Registros:", len(gdf_velocidades_full))
print("Columnas disponibles:")
print(gdf_velocidades_full.columns.tolist())

gdf_velocidades_full.head()

Ruta velocidades: /content/drive/MyDrive/Movilidad_inteligente_madrid/mapa_velocidades/velocidades_madrid_final.geojson
Existe: True
Registros: 271232
Columnas disponibles:
['u', 'v', 'key', 'highway', 'maxspeed', 'es_urbano', 'maxspeed_final', 'geometry']


,u,v,key,highway,maxspeed,es_urbano,maxspeed_final,geometry
0,21741584,21741587,0,[motorway],[100],False,[100],"LINESTRING (-3.69059 40.26517, -3.6905 40.2628..."
1,21741587,759985345,0,[motorway_link],"[40, 70]",False,"[40, 70]","LINESTRING (-3.69004 40.24808, -3.6901 40.2477..."
2,21741587,310025091,0,[motorway],[100],False,[100],"LINESTRING (-3.69004 40.24808, -3.68994 40.245..."
3,21741595,1317348460,0,"[motorway_link, tertiary]",[40],True,[40],"LINESTRING (-3.67595 40.1961, -3.6759 40.19589..."
4,21741595,310031145,0,[motorway],"[100, 120]",False,"[100, 120]","LINESTRING (-3.67595 40.1961, -3.67566 40.1956..."


In [185]:
gdf_velocidades = gdf_velocidades_full.copy()

columnas_necesarias = ["u", "v", "maxspeed_final"]

for col in columnas_necesarias:
    if col not in gdf_velocidades.columns:
        raise ValueError(f"Falta la columna necesaria: {col}")

print("Columnas conservadas:", len(gdf_velocidades.columns))
gdf_velocidades[["u", "v", "maxspeed_final"]].head()

Columnas conservadas: 8


,u,v,maxspeed_final
0,21741584,21741587,[100]
1,21741587,759985345,"[40, 70]"
2,21741587,310025091,[100]
3,21741595,1317348460,[40]
4,21741595,310031145,"[100, 120]"


In [186]:
import ast
import re

def extraer_numeros_velocidad(valor):
    if valor is None:
        return []

    if isinstance(valor, float) and np.isnan(valor):
        return []

    if isinstance(valor, (int, float, np.integer, np.floating)):
        return [float(valor)]

    if isinstance(valor, (list, tuple, set, np.ndarray)):
        numeros = []
        for item in valor:
            numeros.extend(extraer_numeros_velocidad(item))
        return numeros

    texto = str(valor).strip()

    if texto.lower() in ["", "nan", "none"]:
        return []

    try:
        valor_parseado = ast.literal_eval(texto)
        return extraer_numeros_velocidad(valor_parseado)
    except Exception:
        pass

    numeros = re.findall(r"\d+(?:[.,]\d+)?", texto)
    return [float(n.replace(",", ".")) for n in numeros]


def agregar_velocidad(valor, metodo="media"):
    numeros = extraer_numeros_velocidad(valor)
    numeros = [n for n in numeros if 0 < n <= 150]

    if len(numeros) == 0:
        return np.nan

    if metodo == "media":
        return float(np.mean(numeros))
    elif metodo == "mediana":
        return float(np.median(numeros))
    elif metodo == "max":
        return float(np.max(numeros))
    elif metodo == "min":
        return float(np.min(numeros))
    else:
        raise ValueError("Método no válido")


METODO_AGREGACION_VELOCIDAD = "media"

gdf_velocidades["velocidad_kmh"] = gdf_velocidades["maxspeed_final"].apply(
    lambda x: agregar_velocidad(x, metodo=METODO_AGREGACION_VELOCIDAD)
)

print("Filas totales:", len(gdf_velocidades))
print("Velocidades válidas:", gdf_velocidades["velocidad_kmh"].notna().sum())

gdf_velocidades.head()

Filas totales: 271232
Velocidades válidas: 271209


,u,v,key,highway,maxspeed,es_urbano,maxspeed_final,geometry,velocidad_kmh
0,21741584,21741587,0,[motorway],[100],False,[100],"LINESTRING (-3.69059 40.26517, -3.6905 40.2628...",100.0
1,21741587,759985345,0,[motorway_link],"[40, 70]",False,"[40, 70]","LINESTRING (-3.69004 40.24808, -3.6901 40.2477...",55.0
2,21741587,310025091,0,[motorway],[100],False,[100],"LINESTRING (-3.69004 40.24808, -3.68994 40.245...",100.0
3,21741595,1317348460,0,"[motorway_link, tertiary]",[40],True,[40],"LINESTRING (-3.67595 40.1961, -3.6759 40.19589...",40.0
4,21741595,310031145,0,[motorway],"[100, 120]",False,"[100, 120]","LINESTRING (-3.67595 40.1961, -3.67566 40.1956...",110.0


In [187]:
# Preparar u y v
gdf_velocidades["u"] = pd.to_numeric(gdf_velocidades["u"], errors="coerce")
gdf_velocidades["v"] = pd.to_numeric(gdf_velocidades["v"], errors="coerce")

gdf_velocidades_validas = gdf_velocidades.dropna(subset=["u", "v"]).copy()

gdf_velocidades_validas["u"] = gdf_velocidades_validas["u"].astype(int)
gdf_velocidades_validas["v"] = gdf_velocidades_validas["v"].astype(int)

columnas_atributos = [
    col for col in gdf_velocidades_validas.columns
    if col not in ["u", "v", "geometry"]
]

print("Columnas de atributos que se van a conservar:")
print(columnas_atributos)

# Diccionario (u, v) todos los atributos del tramo
atributos_por_arista = (
    gdf_velocidades_validas
    .drop_duplicates(subset=["u", "v"])
    .set_index(["u", "v"])[columnas_atributos]
    .to_dict(orient="index")
)

print("Tramos con atributos disponibles:", len(atributos_por_arista))

Columnas de atributos que se van a conservar:
['key', 'highway', 'maxspeed', 'es_urbano', 'maxspeed_final', 'velocidad_kmh']
Tramos con atributos disponibles: 269506


In [188]:
# Añadir atributos del GeoJSON completo a las aristas de G_osm

aristas_con_atributos = 0
aristas_sin_atributos = 0

for u, v, k, data in G_osm.edges(keys=True, data=True):
    u_int = int(u)
    v_int = int(v)

    atributos = atributos_por_arista.get((u_int, v_int))

    if atributos is None:
        aristas_sin_atributos += 1
        data["velocidad_disponible"] = False
        data["fuente_velocidad"] = "sin_atributos_geojson"
        data["velocidad_kmh"] = np.nan
        data["travel_time_s"] = np.nan
        continue

    aristas_con_atributos += 1


    for nombre_columna, valor in atributos.items():
        if isinstance(valor, (list, tuple, set, dict)):
            valor = str(valor)

        data[f"vel_{nombre_columna}"] = valor

    #velocidad_kmh solo si existe realmente
    velocidad = atributos.get("velocidad_kmh")

    if velocidad is not None and pd.notna(velocidad):
        velocidad = float(velocidad)
        longitud_m = float(data.get("length", 0))
        tiempo_s = longitud_m / (velocidad * 1000 / 3600) if longitud_m > 0 else np.nan

        data["velocidad_kmh"] = velocidad
        data["travel_time_s"] = tiempo_s
        data["fuente_velocidad"] = "geojson_inferencia"
        data["velocidad_disponible"] = True
    else:
        data["velocidad_kmh"] = np.nan
        data["travel_time_s"] = np.nan
        data["fuente_velocidad"] = "sin_velocidad_inferida"
        data["velocidad_disponible"] = False

print("Aristas OSM con atributos del GeoJSON:", aristas_con_atributos)
print("Aristas OSM sin atributos del GeoJSON:", aristas_sin_atributos)

Aristas OSM con atributos del GeoJSON: 61804
Aristas OSM sin atributos del GeoJSON: 39


In [189]:
print("G_osm existe:", "G_osm" in globals())
print("df_pares existe:", "df_pares" in globals())
print("medidores existe:", "medidores" in globals())

print("Aristas G_osm:", G_osm.number_of_edges())
print("Pares candidatos:", len(df_pares))
print("Medidores:", len(medidores))

fuentes_velocidad = [
    data.get("fuente_velocidad")
    for _, _, _, data in G_osm.edges(keys=True, data=True)
]

pd.Series(fuentes_velocidad).value_counts()

G_osm existe: True
df_pares existe: True
medidores existe: True
Aristas G_osm: 61843
Pares candidatos: 60494
Medidores: 5072


,count
geojson_inferencia,61804
sin_atributos_geojson,39


In [190]:
edges_con_tiempo = []

for u, v, k, data in G_osm.edges(keys=True, data=True):
    tiempo = data.get("travel_time_s")

    if tiempo is not None and pd.notna(tiempo) and np.isfinite(float(tiempo)):
        edges_con_tiempo.append((u, v, k))

G_osm_tiempo = G_osm.edge_subgraph(edges_con_tiempo).copy()

print("Aristas originales G_osm:", G_osm.number_of_edges())
print("Aristas con tiempo disponible:", G_osm_tiempo.number_of_edges())
print("Nodos con tiempo disponible:", G_osm_tiempo.number_of_nodes())

Aristas originales G_osm: 61843
Aristas con tiempo disponible: 61804
Nodos con tiempo disponible: 31433


In [191]:
def metricas_camino_osm_tiempo(G, path):

    distancia_total_m = 0.0
    tiempo_total_s = 0.0

    for u, v in zip(path[:-1], path[1:]):
        edges = G.get_edge_data(u, v)

        if edges is None:
            return np.nan, np.nan

        aristas_validas = []

        for data in edges.values():
            tiempo = data.get("travel_time_s")

            if tiempo is not None and pd.notna(tiempo) and np.isfinite(float(tiempo)):
                aristas_validas.append(data)

        if len(aristas_validas) == 0:
            return np.nan, np.nan

        mejor_arista = min(
            aristas_validas,
            key=lambda data: float(data.get("travel_time_s"))
        )

        distancia_total_m += float(mejor_arista.get("length", 0))
        tiempo_total_s += float(mejor_arista.get("travel_time_s", 0))

    return distancia_total_m, tiempo_total_s

In [192]:
df_pares_calculo = df_pares.copy()

print("Pares candidatos a calcular:", len(df_pares_calculo))

Pares candidatos a calcular: 60494


In [193]:
aristas_reales_tiempo = []
pares_sin_camino_tiempo = 0

for _, row in tqdm(df_pares_calculo.iterrows(), total=len(df_pares_calculo)):
    id_origen = int(row["id_origen"])
    id_destino = int(row["id_destino"])

    osm_origen = int(row["osm_node_origen"])
    osm_destino = int(row["osm_node_destino"])

    distancia_directa_m = float(row["distancia_directa_m"])

    if osm_origen == osm_destino:
        aristas_reales_tiempo.append({
            "id_origen": id_origen,
            "id_destino": id_destino,
            "osm_node_origen": osm_origen,
            "osm_node_destino": osm_destino,
            "distancia_directa_m": distancia_directa_m,
            "distancia_red_m": 0.0,
            "tiempo_red_s": 0.0,
            "tiempo_red_min": 0.0,
            "velocidad_media_camino_kmh": np.nan,
            "num_nodos_camino_osm": 1,
            "path_osm": str([osm_origen]),
            "tipo_conexion": "mismo_nodo_osm"
        })
        continue

    try:
        path = nx.shortest_path(
            G_osm_tiempo,
            source=osm_origen,
            target=osm_destino,
            weight="travel_time_s"
        )

        path = [int(n) for n in path]

        distancia_red_m, tiempo_red_s = metricas_camino_osm_tiempo(
            G_osm_tiempo,
            path
        )

        if pd.isna(tiempo_red_s) or tiempo_red_s <= 0:
            pares_sin_camino_tiempo += 1
            continue

        velocidad_media_camino_kmh = distancia_red_m / tiempo_red_s * 3.6

        aristas_reales_tiempo.append({
            "id_origen": id_origen,
            "id_destino": id_destino,
            "osm_node_origen": osm_origen,
            "osm_node_destino": osm_destino,
            "distancia_directa_m": distancia_directa_m,
            "distancia_red_m": distancia_red_m,
            "tiempo_red_s": tiempo_red_s,
            "tiempo_red_min": tiempo_red_s / 60,
            "velocidad_media_camino_kmh": velocidad_media_camino_kmh,
            "num_nodos_camino_osm": len(path),
            "path_osm": str(path),
            "tipo_conexion": "camino_osm_con_tiempo"
        })

    except (nx.NetworkXNoPath, nx.NodeNotFound):
        pares_sin_camino_tiempo += 1
        continue

df_aristas_reales_tiempo = pd.DataFrame(aristas_reales_tiempo).drop_duplicates()

print("Pares candidatos calculados:", len(df_pares_calculo))
print("Aristas reales con tiempo:", len(df_aristas_reales_tiempo))
print("Pares sin camino con tiempo:", pares_sin_camino_tiempo)

df_aristas_reales_tiempo.head()

100%|██████████| 60494/60494 [00:21<00:00, 2786.53it/s]


Pares candidatos calculados: 60494
Aristas reales con tiempo: 60091
Pares sin camino con tiempo: 403


,id_origen,id_destino,osm_node_origen,osm_node_destino,distancia_directa_m,distancia_red_m,tiempo_red_s,tiempo_red_min,velocidad_media_camino_kmh,num_nodos_camino_osm,path_osm,tipo_conexion
0,6835,6833,32636471,32636471,20.104431,0.000000,0.000000,0.000000,NaN,1,[32636471],mismo_nodo_osm
1,6835,6836,32636471,315261895,91.872471,6655.017406,425.102868,7.085048,56.358271,14,"[32636471, 299293137, 21702042, 315518393, 142...",camino_osm_con_tiempo
2,6835,6837,32636471,315264896,93.934596,2248.920678,150.764871,2.512748,53.700271,20,"[32636471, 315266287, 1478929267, 1175412866, ...",camino_osm_con_tiempo
3,6835,6827,32636471,315261895,110.969150,6655.017406,425.102868,7.085048,56.358271,14,"[32636471, 299293137, 21702042, 315518393, 142...",camino_osm_con_tiempo
4,6835,1042,32636471,315265031,116.845927,2583.076553,190.863576,3.181060,48.721059,21,"[32636471, 315266287, 1478929267, 1175412866, ...",camino_osm_con_tiempo


In [194]:
df_aristas_reales_tiempo["factor_rodeo"] = np.where(
    df_aristas_reales_tiempo["distancia_directa_m"] > 0,
    df_aristas_reales_tiempo["distancia_red_m"] / df_aristas_reales_tiempo["distancia_directa_m"],
    np.nan
)

df_aristas_reales_tiempo.loc[
    df_aristas_reales_tiempo["tipo_conexion"] == "mismo_nodo_osm",
    "factor_rodeo"
] = 0

df_aristas_reales_tiempo[
    [
        "distancia_red_m",
        "tiempo_red_s",
        "tiempo_red_min",
        "velocidad_media_camino_kmh",
        "factor_rodeo"
    ]
].describe()

,distancia_red_m,tiempo_red_s,tiempo_red_min,velocidad_media_camino_kmh,factor_rodeo
count,60091.000000,60091.000000,60091.000000,56479.000000,60091.000000
mean,651.355846,50.506896,0.841782,43.185955,4.268884
std,1192.684990,72.850249,1.214171,10.163516,13.005218
min,0.000000,0.000000,0.000000,20.000000,0.000000
25%,191.004896,16.151484,0.269191,34.848282,1.099436
50%,333.994689,30.241137,0.504019,43.656994,1.660860
75%,597.078033,53.610628,0.893510,50.000000,3.209000
max,16339.683383,897.125526,14.952092,90.000000,735.347820


In [195]:
import ast
from shapely.geometry import LineString, MultiLineString

def color_por_velocidad(velocidad):
    if pd.isna(velocidad) or velocidad < 0:
        return "gray"
    elif velocidad <= 20:
        return "darkred"
    elif velocidad <= 30:
        return "red"
    elif velocidad <= 50:
        return "orange"
    elif velocidad <= 70:
        return "green"
    else:
        return "blue"


def mejor_arista_por_tiempo(G, u, v):
    edges = G.get_edge_data(u, v)

    if edges is None:
        return None

    aristas_validas = []

    for data in edges.values():
        tiempo = data.get("travel_time_s")

        if tiempo is not None and pd.notna(tiempo):
            aristas_validas.append(data)

    if len(aristas_validas) == 0:
        return None

    return min(aristas_validas, key=lambda d: float(d.get("travel_time_s")))


def coords_arista_osm(G, u, v):
    data = mejor_arista_por_tiempo(G, u, v)

    if data is None:
        return []

    geom = data.get("geometry")

    if isinstance(geom, LineString):
        return [[lat, lon] for lon, lat in geom.coords]

    elif isinstance(geom, MultiLineString):
        coords = []
        for linea in geom.geoms:
            coords.extend([[lat, lon] for lon, lat in linea.coords])
        return coords

    else:
        lat_u = G.nodes[u]["y"]
        lon_u = G.nodes[u]["x"]
        lat_v = G.nodes[v]["y"]
        lon_v = G.nodes[v]["x"]

        return [[lat_u, lon_u], [lat_v, lon_v]]


def coords_camino_osm(G, path):
    coords_totales = []

    for u, v in zip(path[:-1], path[1:]):
        coords = coords_arista_osm(G, u, v)

        if len(coords) == 0:
            continue

        if len(coords_totales) == 0:
            coords_totales.extend(coords)
        else:
            coords_totales.extend(coords[1:])

    return coords_totales

In [196]:
import folium
from tqdm import tqdm

centro_madrid = [
    medidores["latitud"].mean(),
    medidores["longitud"].mean()
]

mapa_caminos_reales = folium.Map(
    location=centro_madrid,
    zoom_start=12,
    tiles="cartodbpositron",
    prefer_canvas=True
)

df_mapa = df_aristas_reales_tiempo.copy()

print("Aristas a pintar:", len(df_mapa))

rutas_pintadas = 0
rutas_sin_path = 0


for _, row in tqdm(df_mapa.iterrows(), total=len(df_mapa)):
    try:
        if pd.notna(row["path_osm"]):
            path = ast.literal_eval(row["path_osm"])
        else:
            path = nx.shortest_path(
                G_osm_tiempo,
                source=int(row["osm_node_origen"]),
                target=int(row["osm_node_destino"]),
                weight="travel_time_s"
            )

        path = [int(n) for n in path]

        if len(path) <= 1:
            continue

        coords = coords_camino_osm(G_osm_tiempo, path)

        if len(coords) == 0:
            rutas_sin_path += 1
            continue

        velocidad = row["velocidad_media_camino_kmh"]
        tiempo_min = row["tiempo_red_min"]
        distancia_m = row["distancia_red_m"]

        popup = (
            f"<b>Camino real OSM</b><br>"
            f"ID origen: {int(row['id_origen'])}<br>"
            f"ID destino: {int(row['id_destino'])}<br>"
            f"Distancia red: {distancia_m:.1f} m<br>"
            f"Tiempo: {tiempo_min:.2f} min<br>"
            f"Velocidad media: {velocidad:.1f} km/h<br>"
            f"Nodos OSM camino: {int(row['num_nodos_camino_osm'])}"
        )

        folium.PolyLine(
            locations=coords,
            color=color_por_velocidad(velocidad),
            weight=2,
            opacity=0.55,
            popup=popup
        ).add_to(mapa_caminos_reales)

        rutas_pintadas += 1

    except Exception:
        rutas_sin_path += 1
        continue

nodos_en_mapa = set(df_mapa["id_origen"]).union(set(df_mapa["id_destino"]))

print("Nodos medidores a pintar:", len(nodos_en_mapa))

medidores_mapa = medidores[medidores["id"].isin(nodos_en_mapa)].copy()

for _, row in tqdm(medidores_mapa.iterrows(), total=len(medidores_mapa)):
    folium.CircleMarker(
        location=[row["latitud"], row["longitud"]],
        radius=3,
        color="black",
        fill=True,
        fill_color="black",
        fill_opacity=0.9,
        popup=(
            f"<b>Medidor</b><br>"
            f"ID: {int(row['id'])}<br>"
            f"Nombre: {row['nombre']}<br>"
            f"OSM node: {int(row['osm_node'])}"
        )
    ).add_to(mapa_caminos_reales)


leyenda_html = """
<div style="
position: fixed;
bottom: 40px; left: 40px; width: 270px; height: 210px;
background-color: white;
border:2px solid grey;
z-index:9999;
font-size:14px;
padding: 10px;
">
<b>Grafo de medidores</b><br>
<span style="color:black;">●</span> Nodo medidor<br><br>

<b>Velocidad media del camino OSM</b><br>
<span style="color:darkred;">━━</span> ≤ 20 km/h<br>
<span style="color:red;">━━</span> 21 - 30 km/h<br>
<span style="color:orange;">━━</span> 31 - 50 km/h<br>
<span style="color:green;">━━</span> 51 - 70 km/h<br>
<span style="color:blue;">━━</span> > 70 km/h<br>
<span style="color:gray;">━━</span> sin dato<br>
</div>
"""

mapa_caminos_reales.get_root().html.add_child(folium.Element(leyenda_html))

ruta_mapa_caminos_reales = PROCESSED_DIR / "mapa_caminos_reales_osm_medidores_completo.html"

mapa_caminos_reales.save(ruta_mapa_caminos_reales)

print("Rutas pintadas:", rutas_pintadas)
print("Rutas no pintadas:", rutas_sin_path)
print("Mapa guardado en:", ruta_mapa_caminos_reales)
print("Existe:", ruta_mapa_caminos_reales.exists())



Aristas a pintar: 60091


100%|██████████| 60091/60091 [00:36<00:00, 1642.42it/s]


Nodos medidores a pintar: 5060


100%|██████████| 5060/5060 [00:00<00:00, 8518.04it/s]


Rutas pintadas: 56479
Rutas no pintadas: 0
Mapa guardado en: /content/drive/MyDrive/Movilidad_inteligente_madrid/data/processed/mapa_caminos_reales_osm_medidores_completo.html
Existe: True


COMPARATIVA DE POR QUÉ SALEN ARISTAS DIFERENTES

In [197]:
from datetime import datetime

print("Ruta GeoJSON:", ruta_velocidades)
print("Existe GeoJSON:", ruta_velocidades.exists())

fecha_modificacion_geojson = datetime.fromtimestamp(
    ruta_velocidades.stat().st_mtime
)

print("Fecha modificación/subida GeoJSON:", fecha_modificacion_geojson)

print("Metadatos G_osm:")
print(G_osm.graph)

Ruta GeoJSON: /content/drive/MyDrive/Movilidad_inteligente_madrid/mapa_velocidades/velocidades_madrid_final.geojson
Existe GeoJSON: True
Fecha modificación/subida GeoJSON: 2026-06-17 07:52:26
Metadatos G_osm:
{'created_date': '2026-06-22 11:24:26', 'created_with': 'OSMnx 2.1.0', 'crs': 'epsg:4326', 'simplified': True}


In [198]:
aristas_osm_actuales = []

for u, v, k, data in G_osm.edges(keys=True, data=True):
    aristas_osm_actuales.append({
        "u": int(u),
        "v": int(v),
        "key": int(k),
        "osmid": str(data.get("osmid")),
        "name": str(data.get("name")),
        "highway": str(data.get("highway")),
        "length": data.get("length"),
        "fuente_velocidad": data.get("fuente_velocidad"),
        "velocidad_kmh": data.get("velocidad_kmh"),
        "travel_time_s": data.get("travel_time_s")
    })

df_edges_osm_actual = pd.DataFrame(aristas_osm_actuales)

print("Aristas actuales en G_osm:", len(df_edges_osm_actual))

df_edges_osm_actual.head()

Aristas actuales en G_osm: 61843


,u,v,key,osmid,name,highway,length,fuente_velocidad,velocidad_kmh,travel_time_s
0,171946,26513145,0,807334397,Calle de Velázquez,secondary,43.951501,geojson_inferencia,50.0,3.164508
1,171951,1209331009,0,104864843,Calle Juan de Mena,residential,18.732956,geojson_inferencia,30.0,2.247955
2,171951,26486636,0,553113575,Calle de Alfonso XII,secondary,85.916903,geojson_inferencia,50.0,6.186017
3,171952,26486617,0,317302445,Calle de Alfonso XII,secondary,29.284676,geojson_inferencia,50.0,2.108497
4,171953,2681222064,0,28401583,Calle Espalter,residential,21.168811,geojson_inferencia,30.0,2.540257


In [199]:
# Usamos el GeoJSON completo que ya cargaste
gdf_vel_check = gdf_velocidades_full.copy()

gdf_vel_check["u"] = pd.to_numeric(gdf_vel_check["u"], errors="coerce")
gdf_vel_check["v"] = pd.to_numeric(gdf_vel_check["v"], errors="coerce")

gdf_vel_check = gdf_vel_check.dropna(subset=["u", "v"]).copy()

gdf_vel_check["u"] = gdf_vel_check["u"].astype(int)
gdf_vel_check["v"] = gdf_vel_check["v"].astype(int)

df_edges_geojson = gdf_vel_check[["u", "v"]].drop_duplicates().copy()

print("Aristas únicas en GeoJSON de velocidades:", len(df_edges_geojson))

df_edges_geojson.head()

Aristas únicas en GeoJSON de velocidades: 269506


,u,v
0,21741584,21741587
1,21741587,759985345
2,21741587,310025091
3,21741595,1317348460
4,21741595,310031145


In [200]:
set_osm_actual = set(
    zip(df_edges_osm_actual["u"], df_edges_osm_actual["v"])
)

set_geojson = set(
    zip(df_edges_geojson["u"], df_edges_geojson["v"])
)

osm_sin_geojson = set_osm_actual - set_geojson
geojson_sin_osm = set_geojson - set_osm_actual
coinciden = set_osm_actual & set_geojson

print("Aristas G_osm actual:", len(set_osm_actual))
print("Aristas GeoJSON velocidades:", len(set_geojson))
print("Coinciden por (u, v):", len(coinciden))
print("Están en G_osm actual pero NO en GeoJSON:", len(osm_sin_geojson))
print("Están en GeoJSON pero NO en G_osm actual:", len(geojson_sin_osm))

Aristas G_osm actual: 61559
Aristas GeoJSON velocidades: 269506
Coinciden por (u, v): 61520
Están en G_osm actual pero NO en GeoJSON: 39
Están en GeoJSON pero NO en G_osm actual: 207986


In [201]:
revision_osm_sin_geojson = []

for u, v in osm_sin_geojson:
    existe_inverso = (v, u) in set_geojson

    revision_osm_sin_geojson.append({
        "u": u,
        "v": v,
        "existe_en_geojson_directo": False,
        "existe_en_geojson_inverso": existe_inverso,
        "diagnostico": "existe_inverso" if existe_inverso else "no_existe_en_geojson"
    })

df_revision_osm_sin_geojson = pd.DataFrame(revision_osm_sin_geojson)

print("Aristas OSM sin match en GeoJSON:", len(df_revision_osm_sin_geojson))

df_revision_osm_sin_geojson["diagnostico"].value_counts()

df_revision_osm_sin_geojson.head(50)

Aristas OSM sin match en GeoJSON: 39


,u,v,existe_en_geojson_directo,existe_en_geojson_inverso,diagnostico
0,13953905230,354928368,False,False,no_existe_en_geojson
1,13052424211,13953905201,False,False,no_existe_en_geojson
2,442632874,13953905230,False,False,no_existe_en_geojson
3,965442011,1812894222,False,False,no_existe_en_geojson
4,571482450,3158288085,False,True,existe_inverso
5,1982834376,25906975,False,False,no_existe_en_geojson
6,13665106168,13665106169,False,False,no_existe_en_geojson
7,316637698,317567808,False,False,no_existe_en_geojson
8,25906978,1505542782,False,False,no_existe_en_geojson
9,442628204,442632874,False,False,no_existe_en_geojson


In [202]:
df_osm_sin_match_detalle = df_edges_osm_actual.merge(
    df_revision_osm_sin_geojson,
    on=["u", "v"],
    how="inner"
)

print("Aristas OSM actuales sin match en GeoJSON:", len(df_osm_sin_match_detalle))

df_osm_sin_match_detalle[
    [
        "u", "v", "key", "osmid", "name", "highway",
        "length", "diagnostico"
    ]
].head(50)

Aristas OSM actuales sin match en GeoJSON: 39


,u,v,key,osmid,name,highway,length,diagnostico
0,21994228,13949072791,0,147157053,Calle República Checa,residential,81.495059,no_existe_en_geojson
1,25906978,1505542782,0,"[1453554752, 75336312]",Paseo de la Castellana,"['trunk', 'trunk_link']",150.981486,no_existe_en_geojson
2,25908512,13953905230,0,71045417,Paseo de la Castellana,secondary,14.343175,no_existe_en_geojson
3,31031060,248001162,0,208914108,Calle de Pedro Laborde,residential,69.000886,no_existe_en_geojson
4,137450258,13953905202,0,1531015358,None,busway,187.648253,no_existe_en_geojson
5,137452400,13953905201,0,298699352,Plaza de Andrés Manjón,residential,2.467973,no_existe_en_geojson
6,137485778,13953905202,0,14306536,Plaza de Andrés Manjón,residential,53.911366,no_existe_en_geojson
7,307644015,316637757,0,28016640,Calle Sierra Faladora,residential,107.627581,no_existe_en_geojson
8,316637698,317567808,0,28798587,Calle Pedro Callejo,residential,88.226593,no_existe_en_geojson
9,317567786,1250622590,0,93519938,Calle Adra,residential,98.136112,no_existe_en_geojson


In [203]:
ruta_grafo_osm_base = GRAPH_DIR / "red_osm_madrid_base.graphml"

ox.save_graphml(
    G_osm,
    filepath=ruta_grafo_osm_base
)

print("Grafo OSM guardado en:", ruta_grafo_osm_base)
print("Existe:", ruta_grafo_osm_base.exists())

Grafo OSM guardado en: /content/drive/MyDrive/Movilidad_inteligente_madrid/data/graphs/red_osm_madrid_base.graphml
Existe: True


In [204]:
ruta_grafo_osm_base = GRAPH_DIR / "red_osm_madrid_base.graphml"

G_osm = ox.load_graphml(ruta_grafo_osm_base)

print("Grafo OSM cargado desde:", ruta_grafo_osm_base)
print("Nodos OSM:", G_osm.number_of_nodes())
print("Aristas OSM:", G_osm.number_of_edges())

Grafo OSM cargado desde: /content/drive/MyDrive/Movilidad_inteligente_madrid/data/graphs/red_osm_madrid_base.graphml
Nodos OSM: 31443
Aristas OSM: 61843
